# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/124pritivarma6001-commits/flyrank_internship_ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I first inspected the distributions of impressions, CTR, and average position at content level. The purpose was to understand the scale and spread of the signals before testing the assumptions behind the baseline rule. The distributions are uneven, especially for impressions and CTR, so simple averages should be interpreted carefully.

In [14]:
%pip -q install duckdb huggingface_hub

In [15]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


In [16]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [17]:
# Section 1: Distributions

content_signals = con.sql(f"""
WITH content_level AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(
            CASE
                WHEN gsc_impressions > 0
                THEN gsc_avg_position * gsc_impressions
                ELSE 0
            END
        ) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM {TABLES["fact_daily"]}
    WHERE gsc_impressions IS NOT NULL
      AND gsc_clicks IS NOT NULL
      AND gsc_avg_position IS NOT NULL
    GROUP BY content_hash_id
),

base AS (
    SELECT
        *,
        CAST(clicks AS DOUBLE) / NULLIF(impressions, 0) AS ctr
    FROM content_level
    WHERE impressions > 0
)

SELECT *
FROM base
""").df()

print(f"Total content items: {len(content_signals):,}")

print("\nSummary statistics:")
display(
    content_signals[
        ["impressions", "clicks", "ctr", "avg_position"]
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

print("\nZero-click content:")
zero_clicks = (content_signals["clicks"] == 0).sum()
print(f"{zero_clicks:,} / {len(content_signals):,}")

print("\nBaseline thresholds:")
print("Impressions threshold :", 237)
print("CTR threshold         :", 0.003026)
print("Position threshold    :", 20)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total content items: 309,234

Summary statistics:


,impressions,clicks,ctr,avg_position
count,3.092340e+05,309234.000000,309234.000000,309234.000000
mean,5.702857e+03,20.828150,0.006333,17.666185
std,2.513556e+04,396.757832,0.028146,18.498427
min,1.000000e+00,0.000000,0.000000,0.000000
25%,2.000000e+01,0.000000,0.000000,5.875000
50%,2.370000e+02,0.000000,0.000000,10.000000
75%,2.282000e+03,5.000000,0.003024,22.668292
90%,1.186600e+04,33.000000,0.008734,46.037141
95%,2.673570e+04,84.000000,0.021208,63.435884
99%,9.065567e+04,344.000000,0.100000,76.506118



Zero-click content:
160,293 / 309,234

Baseline thresholds:
Impressions threshold : 237
CTR threshold         : 0.003026
Position threshold    : 20


## 2. Signal test #1 / #2 / #3 (verdict each)

I tested three signals used by the baseline rule: impressions, CTR, and average position. Each test compares groups of content rather than relying only on a single correlation value. The verdicts are based on the observed direction in the data and are treated as directional evidence, not causal proof.

In [21]:
# Section 2: Signal audit
df = content_signals.copy()
import pandas as pd
import numpy as np

# -----------------------------
# SIGNAL #1 — IMPRESSIONS
# -----------------------------
df["impressions_bucket"] = pd.qcut(
    df["impressions"],
    q=4,
    duplicates="drop"
)

impression_audit = (
    df.groupby("impressions_bucket", observed=True)
      .agg(
          contents=("content_hash_id", "count"),
          median_impressions=("impressions", "median"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("SIGNAL #1 — IMPRESSIONS")
print(impression_audit)


# -----------------------------
# SIGNAL #2 — CTR
# -----------------------------
ctr_threshold = 0.003026

df["ctr_group"] = np.where(
    df["ctr"] < ctr_threshold,
    "Low CTR",
    "High CTR"
)

ctr_audit = (
    df.groupby("ctr_group")
      .agg(
          contents=("content_hash_id", "count"),
          median_ctr=("ctr", "median"),
          median_impressions=("impressions", "median"),
          median_position=("avg_position", "median")
      )
      .reset_index()
)

print("\nSIGNAL #2 — CTR")
print(ctr_audit)

low_ctr_pct = (df["ctr"] < ctr_threshold).mean() * 100

print(
    f"\nContent below baseline CTR threshold: "
    f"{low_ctr_pct:.2f}%"
)


# -----------------------------
# SIGNAL #3 — AVERAGE POSITION
# -----------------------------
df["position_bucket"] = pd.cut(
    df["avg_position"],
    bins=[0, 5, 10, 20, 50, np.inf],
    labels=["1-5", "5-10", "10-20", "20-50", "50+"]
)

position_audit = (
    df.groupby("position_bucket", observed=True)
      .agg(
          contents=("content_hash_id", "count"),
          median_position=("avg_position", "median"),
          median_ctr=("ctr", "median"),
          mean_ctr=("ctr", "mean")
      )
      .reset_index()
)

print("\nSIGNAL #3 — AVERAGE POSITION")
print(position_audit)

good_position_pct = (
    (df["avg_position"] <= 20).mean() * 100
)

print(
    f"\nContent with avg position <=20: "
    f"{good_position_pct:.2f}%"
)


# -----------------------------
# VERDICTS
# -----------------------------
print("\nVERDICTS")
print("Signal #1 — Impressions: CONFIRMED")
print("Signal #2 — CTR: CONFIRMED")
print("Signal #3 — Average position: MIXED")

SIGNAL #1 — IMPRESSIONS
    impressions_bucket  contents  median_impressions  median_ctr  mean_ctr
0        (0.999, 20.0]     83582                 5.0    0.000000  0.013831
1        (20.0, 237.0]     71167                76.0    0.000000  0.004444
2      (237.0, 2282.0]     77185               740.0    0.001502  0.002942
3  (2282.0, 2902616.0]     77300              8584.0    0.002188  0.003350

SIGNAL #2 — CTR
  ctr_group  contents  median_ctr  median_impressions  median_position
0  High CTR     77282    0.006904               892.0         8.860168
1   Low CTR    231952    0.000000               123.0        10.580645

Content below baseline CTR threshold: 75.01%

SIGNAL #3 — AVERAGE POSITION
  position_bucket  contents  median_position  median_ctr  mean_ctr
0             1-5     45874         3.800000    0.000000  0.009777
1            5-10     95268         7.181767    0.001094  0.005234
2           10-20     67063        13.738873    0.000764  0.003818
3           20-50     60712

## 3. The flag-linked test

I used average position as the flag-linked signal because the refresh logic relies on search visibility. I tested whether content with good average position also shows the low-CTR pattern assumed by the baseline rule. The result is treated as directional evidence for the rule, not as proof that position causes CTR.

In [22]:
# Section 3: Flag-linked test

ctr_threshold = 0.003026
position_threshold = 20

df["low_ctr"] = df["ctr"] < ctr_threshold
df["good_position"] = df["avg_position"] <= position_threshold

position_test = (
    df.groupby("good_position")
      .agg(
          contents=("content_hash_id", "count"),
          mean_ctr=("ctr", "mean"),
          median_ctr=("ctr", "median"),
          pct_low_ctr=("low_ctr", "mean"),
          mean_impressions=("impressions", "mean")
      )
      .reset_index()
)

position_test["pct_low_ctr"] *= 100

print("FLAG-LINKED TEST — AVERAGE POSITION")
print(position_test)


# Compare low-CTR prevalence
good_position_rate = (
    df.loc[df["good_position"], "low_ctr"].mean() * 100
)

poor_position_rate = (
    df.loc[~df["good_position"], "low_ctr"].mean() * 100
)

print(
    f"\nLow-CTR rate at position <=20: "
    f"{good_position_rate:.2f}%"
)

print(
    f"Low-CTR rate at position >20: "
    f"{poor_position_rate:.2f}%"
)


# Verdict
if good_position_rate < poor_position_rate:
    print("\nVERDICT: CONFIRMED")
    print(
        "Observed data supports the expected direction: "
        "better position is associated with lower low-CTR prevalence."
    )
elif good_position_rate > poor_position_rate:
    print("\nVERDICT: OPPOSITE")
    print(
        "Observed data moves opposite to the expected direction."
    )
else:
    print("\nVERDICT: MIXED")
    print(
        "Observed data does not show a clear directional difference."
    )

FLAG-LINKED TEST — AVERAGE POSITION
   good_position  contents  mean_ctr  median_ctr  pct_low_ctr  \
0          False     87399  0.008281    0.000000    81.728624   
1           True    221835  0.005565    0.000452    72.360989   

   mean_impressions  
0       2721.729162  
1       6877.367422  

Low-CTR rate at position <=20: 72.36%
Low-CTR rate at position >20: 81.73%

VERDICT: CONFIRMED
Observed data supports the expected direction: better position is associated with lower low-CTR prevalence.


## 4. What this means in practice

The signal audit shows how impressions, CTR, and average position behave across the observed content population. The results provide directional decision-support for prioritizing content review, but they do not prove that changing a page will cause CTR to increase. Content teams should therefore use the signals to prioritize human review rather than treat the rule as an automatic decision.

In [23]:
# Section 4: Practical summary

print("Practical summary of the signal audit")
print("--------------------------------------")

print(f"Total content items analysed: {len(df):,}")

low_ctr_count = (
    df["ctr"] < 0.003026
).sum()

good_position_count = (
    df["avg_position"] <= 20
).sum()

both_conditions = (
    (df["ctr"] < 0.003026) &
    (df["avg_position"] <= 20)
).sum()

print(
    f"Content with CTR below 0.003026: "
    f"{low_ctr_count:,}"
)

print(
    f"Content with average position <= 20: "
    f"{good_position_count:,}"
)

print(
    f"Content meeting both low-CTR and good-position conditions: "
    f"{both_conditions:,}"
)

print("\nSignals audited:")
print("- Impressions")
print("- CTR")
print("- Average position")

print("\nPractical interpretation:")
print(
    "The audited signals can support prioritisation and human review, "
    "but they should not be treated as automatic decisions."
)

Practical summary of the signal audit
--------------------------------------
Total content items analysed: 309,234
Content with CTR below 0.003026: 231,952
Content with average position <= 20: 221,835
Content meeting both low-CTR and good-position conditions: 160,522

Signals audited:
- Impressions
- CTR
- Average position

Practical interpretation:
The audited signals can support prioritisation and human review, but they should not be treated as automatic decisions.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.